# 01 : exploration

Chargement et inspection des deux jeux de données, avant de toucher à quoi que ce soit. Chaque constat devient une décision de nettoyage, appliquée dans `02_nettoyage.ipynb`.

## Questions

1. Comment la part de l'IA dans les sujets populaires de Hacker News a-t-elle évolué mois par mois depuis 2020 ? Y a-t-il une cassure à la sortie de ChatGPT (novembre 2022) ?
2. Les articles sur l'IA sont-ils mieux notés et plus commentés que les autres ?
3. Les sorties de modèles des grands labos (OpenAI, Anthropic, Google, Meta) provoquent-elles un pic d'articles dans les jours qui suivent ?
4. Quels labos et quels pays publient le plus de modèles, et la part des modèles à poids ouverts progresse-t-elle ?
5. À quelle vitesse la puissance de calcul des modèles augmente-t-elle d'une année à l'autre ?

## 1. Chargement

Les dates sont typées dès la lecture : sans vraie date, aucun calcul par mois ou par jour n'est possible.

In [1]:
import numpy as np
import pandas as pd

articles = pd.read_csv(
    "../data/raw/hacker_news_top_stories_2020_2026.csv",
    parse_dates=["created_at"],
)
modeles = pd.read_csv(
    "../data/raw/ai_models_panel.csv",
    parse_dates=["publication_date", "last_modified"],
)

print(f"Articles Hacker News : {articles.shape}")
print(f"Modèles d'IA : {modeles.shape}")

Articles Hacker News : (100286, 11)
Modèles d'IA : (3626, 28)


## 2. Hacker News : les articles populaires

In [2]:
articles.head()

,id,title,url,domain,author,score,num_comments,created_at,created_at_i,story_type,year
0,21927248,Rewriting m4vgalib in Rust,http://cliffle.com/blog/m4vga-in-rust/,cliffle.com,zimmerfrei,372,44,2020-01-01 08:38:20+00:00,1577867900,story,2020
1,21927322,What Makes a Successful Founder?,https://www.basisset.ventures/founder-superpowers,basisset.ventures,yarapavan,109,46,2020-01-01 09:07:52+00:00,1577869672,story,2020
2,21927427,I'm not feeling the async pressure,https://lucumr.pocoo.org/2020/1/1/async-pressure/,lucumr.pocoo.org,pauloxnet,337,100,2020-01-01 10:00:04+00:00,1577872804,story,2020
3,21927473,Hard deadlines are not user-first,http://jatins.gitlab.io/me/why-deadline/,jatins.gitlab.io,jatins,467,246,2020-01-01 10:22:42+00:00,1577874162,story,2020
4,21927773,What Happened in the 2010s,https://avc.com/2019/12/what-happened-in-the-2...,avc.com,notlukesky,167,90,2020-01-01 12:26:04+00:00,1577881564,story,2020


In [3]:
articles.info()

<class 'pandas.DataFrame'>
RangeIndex: 100286 entries, 0 to 100285
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   id            100286 non-null  int64              
 1   title         100286 non-null  str                
 2   url           96820 non-null   str                
 3   domain        96820 non-null   str                
 4   author        100286 non-null  str                
 5   score         100286 non-null  int64              
 6   num_comments  100286 non-null  int64              
 7   created_at    100286 non-null  datetime64[us, UTC]
 8   created_at_i  100286 non-null  int64              
 9   story_type    100286 non-null  str                
 10  year          100286 non-null  int64              
dtypes: datetime64[us, UTC](1), int64(5), str(5)
memory usage: 8.4 MB


In [4]:
articles.describe().round(1)

,id,score,num_comments,created_at_i,year
count,100286.0,100286.0,100286.0,1.002860e+05,100286.0
mean,34515685.9,264.2,160.2,1.675003e+09,2022.6
std,7527973.2,227.9,175.6,5.646999e+07,1.8
min,21927248.0,100.0,0.0,1.577868e+09,2020.0
25%,27931558.2,136.0,59.0,1.627053e+09,2021.0
50%,34413928.0,192.0,112.0,1.673971e+09,2023.0
75%,40961277.8,303.0,200.0,1.720969e+09,2024.0
max,47892019.0,5710.0,9274.0,1.777046e+09,2026.0


In [5]:
print(f"Du {articles['created_at'].min()} au {articles['created_at'].max()}")
articles["created_at"].dt.year.value_counts().sort_index()

Du 2020-01-01 08:38:20+00:00 au 2026-04-24 15:59:19+00:00


created_at
2020    15778
2021    16624
2022    16908
2023    17767
2024    14068
2025    14204
2026     4937
Name: count, dtype: int64

Le jeu couvre du 1er janvier 2020 au 24 avril 2026, avec 14 000 à 18 000 articles par an. 2026 n'est couverte que sur moins de 4 mois : les comparaisons entre années devront se faire en part, pas en nombre d'articles.

Le score minimum est exactement 100 : le jeu ne garde que les articles ayant atteint 100 points. Les scores et les commentaires sont très étirés vers le haut (score médian 192, maximum 5 710).

### Valeurs manquantes et colonnes inutiles

In [6]:
articles.isna().sum()

id                 0
title              0
url             3466
domain          3466
author             0
score              0
num_comments       0
created_at         0
created_at_i       0
story_type         0
year               0
dtype: int64

In [7]:
articles_sans_url = articles[articles["url"].isna()]
articles_sans_url["title"].str[:20].value_counts().head()

title
Ask HN: Who is hirin    77
Ask HN: Who wants to    73
Ask HN: What are som    37
Ask HN: What are you    36
Ask HN: What is the     32
Name: count, dtype: int64

In [8]:
print(f"Valeurs de story_type : {articles['story_type'].unique().tolist()}")

date_depuis_timestamp = pd.to_datetime(articles["created_at_i"], unit="s", utc=True)
print(f"created_at_i identique à created_at : {(date_depuis_timestamp == articles['created_at']).mean():.0%}")
print(f"year identique à l'année de created_at : {(articles['year'] == articles['created_at'].dt.year).mean():.0%}")

Valeurs de story_type : ['story']
created_at_i identique à created_at : 100%
year identique à l'année de created_at : 100%


- `url` et `domain` manquent pour les mêmes 3 466 articles : ce sont des messages écrits directement sur Hacker News (« Ask HN: Who is hiring? »…), sans lien externe. L'absence est une information, pas une erreur.
- `story_type` ne prend qu'une seule valeur : la colonne n'apporte rien.
- `created_at_i` (le timestamp) et `year` répètent `created_at` à 100 % : elles sont redondantes.

### Doublons

In [9]:
print(f"Lignes entièrement identiques : {articles.duplicated().sum()}")
print(f"Identifiants en double : {articles['id'].duplicated().sum()}")
print(f"Titres en double : {articles['title'].duplicated().sum()}")
print(f"URL en double : {articles['url'].dropna().duplicated().sum()}")

Lignes entièrement identiques : 0
Identifiants en double : 0
Titres en double : 949
URL en double : 2000


In [10]:
titres_repetes = articles[articles["title"].duplicated(keep=False)]
titres_repetes.sort_values(["title", "created_at"])[["title", "created_at", "score"]].head(6)

,title,created_at,score
51785,"""BGP at home"": getting a DIA circuit installed...",2023-02-19 01:31:24+00:00,183
80346,"""BGP at home"": getting a DIA circuit installed...",2024-12-07 23:45:11+00:00,242
73347,'I was misidentified as shoplifter by facial r...,2024-05-26 00:45:04+00:00,144
73355,'I was misidentified as shoplifter by facial r...,2024-05-26 07:08:26+00:00,160
18831,-2000 Lines of Code (2004),2021-03-08 16:07:05+00:00,646
46590,-2000 Lines of Code (2004),2022-11-05 16:48:58+00:00,332


Aucune ligne n'est entièrement dupliquée et chaque `id` est unique. En revanche, 949 titres et 2 000 URL reviennent plusieurs fois : ce sont des articles republiés, parfois à plusieurs années d'écart. Chaque publication a eu sa propre vie (son score, ses commentaires), mais les compter toutes gonflerait la place d'un sujet.

### Domaines et URL

In [11]:
articles['domain'].value_counts().head(10)

domain
github.com         7026
twitter.com        1750
arstechnica.com     955
theguardian.com     889
reuters.com         871
youtube.com         862
bbc.com             802
nytimes.com         770
arxiv.org           733
bloomberg.com       618
Name: count, dtype: int64

In [12]:
urls_avec_parametres = articles["url"].str.contains("?", regex=False, na=False)
print(f"URL avec des paramètres après le ? : {urls_avec_parametres.sum()}")

URL avec des paramètres après le ? : 2657


2 657 URL contiennent des paramètres de requête. La plupart servent au suivi publicitaire, mais l'un d'eux est plus gênant : la ligne 759 du fichier brut contient un lien de téléchargement ScienceDirect avec des identifiants AWS temporaires d'Elsevier (expirés depuis 2020), repérés par l'analyse de secrets de GitHub.

## 3. Epoch AI : les modèles d'IA

In [13]:
modeles.head()

,model_name,organization,organization_type,country,publication_date,publication_year,domain,task,parameters,training_compute_flop,...,training_code_accessibility,notability_criteria,citations,confidence,is_frontier_model,is_foundation_model,possibly_over_1e23_flop,base_model,reference_link,last_modified
0,Theseus,Bell Laboratories,Industry,United States of America,1950-07-02,1950.0,Robotics,Maze solving,40.0,40.0,...,NaN,Historical significance,NaN,Confident,1,0,0,NaN,https://www.technologyreview.com/2018/12/19/13...,2025-11-28
1,SNARC,Harvard University,Academia,United States of America,1952-01-08,1952.0,Robotics,Maze solving,40.0,NaN,...,NaN,Historical significance,33.0,NaN,0,0,0,NaN,https://en.wikipedia.org/wiki/Stochastic_neura...,2025-11-28
2,Genetic algorithm,Institute for Advanced Study,Academia,United States of America,1954-07-02,1954.0,Mathematics;Biology,Numerical simulation,NaN,NaN,...,NaN,Historical significance,266.0,Unknown,0,0,0,NaN,https://link.springer.com/article/10.1007/BF01...,2025-11-28
3,Self Organizing System,Massachusetts Institute of Technology (MIT),Academia,United States of America,1955-03-01,1955.0,Other,Pattern recognition,225.0,NaN,...,NaN,Historical significance,93.0,NaN,0,0,0,NaN,https://dl.acm.org/doi/10.1145/1455292.1455309,2025-11-28
4,Sequence-based pattern recognition,Massachusetts Institute of Technology (MIT),Academia,United States of America,1955-03-01,1955.0,Vision,Character recognition (OCR),NaN,NaN,...,NaN,Historical significance,290.0,Unknown,0,0,0,NaN,https://dl.acm.org/doi/10.1145/1455292.1455310,2025-11-28


In [14]:
modeles.info()

<class 'pandas.DataFrame'>
RangeIndex: 3626 entries, 0 to 3625
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   model_name                   3626 non-null   str           
 1   organization                 3551 non-null   str           
 2   organization_type            3527 non-null   str           
 3   country                      3539 non-null   str           
 4   publication_date             3614 non-null   datetime64[us]
 5   publication_year             3614 non-null   float64       
 6   domain                       3553 non-null   str           
 7   task                         3523 non-null   str           
 8   parameters                   2344 non-null   float64       
 9   training_compute_flop        1410 non-null   float64       
 10  training_compute_log10_flop  1410 non-null   float64       
 11  training_time_hours          556 non-null    float64  

In [15]:
modeles.describe().round(2).T

,count,mean,min,25%,50%,75%,max,std
publication_date,3614,2021-02-15 02:11:29.319313,1950-07-02 00:00:00,2020-06-30 00:00:00,2023-11-02 00:00:00,2024-10-22 00:00:00,2026-09-30 00:00:00,NaN
publication_year,3614.0,2020.61,1950.0,2020.0,2023.0,2024.0,2026.0,8.09
parameters,2344.0,188022417244.329987,10.0,107617500.0,2920000000.0,14800000000.0,173900000000000.0,4159629292213.350098
training_compute_flop,1410.0,2417745636885145451495424.0,40.0,12343710937500000256.0,4228941549999999680512.0,160000000000000000000000.0,1000099999999999966202298368.0,32970760220494427144060928.0
training_compute_log10_flop,1410.0,20.75,1.6,19.09,21.63,23.2,27.0,3.45
training_time_hours,556.0,483.64,0.1,39.25,148.6,504.0,9022.8,918.2
hardware_quantity,853.0,1189.82,1.0,5.0,32.0,256.0,200000.0,8791.07
training_power_draw_w,775.0,1104327.96,75.83,4190.25,28374.69,203998.55,232668778.58,9664931.3
training_compute_cost_usd,225.0,6159457.91,3.9,1926.9,29374.47,239188.69,387842678.09,38604502.06
citations,1484.0,4573.55,0.0,56.0,493.0,2718.0,228517.0,15773.74


In [16]:
print(f"Du {modeles['publication_date'].min().date()} au {modeles['publication_date'].max().date()}")
print(f"Modèles publiés depuis 2020 : {(modeles['publication_date'] >= '2020-01-01').sum()}")

Du 1950-07-02 au 2026-09-30
Modèles publiés depuis 2020 : 2774


3 626 modèles, du tout premier (Theseus, 1950) à septembre 2026, dont 2 774 publiés depuis 2020 : la période commune avec Hacker News concentre l'essentiel du jeu.

Les mesures techniques s'étalent sur des dizaines d'ordres de grandeur : la puissance de calcul va de 40 à 10²⁷ FLOP, les paramètres de 10 à 174 000 milliards. Une moyenne n'a pas de sens sur de telles échelles : il faudra travailler en échelle logarithmique.

### Valeurs manquantes

In [17]:
(modeles.isna().mean() * 100).round(1).sort_values(ascending=False)

training_compute_cost_usd      93.8
training_time_hours            84.7
base_model                     80.9
training_power_draw_w          78.6
hardware_quantity              76.5
notability_criteria            69.6
training_hardware              67.2
training_compute_flop          61.1
training_compute_log10_flop    61.1
citations                      59.1
parameters                     35.4
training_code_accessibility    33.4
open_model_weights             23.4
model_accessibility            23.3
confidence                      5.4
task                            2.8
organization_type               2.7
country                         2.4
organization                    2.1
domain                          2.0
reference_link                  1.1
publication_date                0.3
publication_year                0.3
model_name                      0.0
is_foundation_model             0.0
is_frontier_model               0.0
possibly_over_1e23_flop         0.0
last_modified               

Les manquants sont massifs sur les mesures d'entraînement : 94 % pour le coût, 85 % pour la durée, 61 % pour la puissance de calcul, 35 % pour les paramètres. Ces chiffres ne sont tout simplement pas publiés pour beaucoup de modèles, surtout les modèles fermés. Les questions 4 et 5 ne pourront porter que sur les modèles renseignés, et il faudra le dire.

12 modèles n'ont aucune date de publication : ils ne peuvent pas entrer dans une analyse dans le temps.

### Colonnes redondantes

In [18]:
pd.crosstab(modeles["model_accessibility"].fillna("manquant"), modeles["open_model_weights"].fillna("manquant"))

open_model_weights,No,Yes,manquant
model_accessibility,,,
API access,437,0,0
Hosted access (no API),133,0,0
Limited access,0,0,3
Open weights (non-commercial),0,229,0
Open weights (restricted use),0,301,0
Open weights (unrestricted),0,846,0
Unreleased,833,0,0
manquant,0,0,844


In [19]:
compute_connu = modeles["training_compute_flop"].notna()
ecart_log = np.log10(modeles.loc[compute_connu, "training_compute_flop"]) - modeles.loc[compute_connu, "training_compute_log10_flop"]
dates_connues = modeles["publication_date"].notna()
annee_coherente = modeles.loc[dates_connues, "publication_year"] == modeles.loc[dates_connues, "publication_date"].dt.year

print(f"training_compute_log10_flop = log10(training_compute_flop) : {(ecart_log.abs() < 0.01).mean():.0%}")
print(f"publication_year = année de publication_date : {annee_coherente.mean():.0%}")

training_compute_log10_flop = log10(training_compute_flop) : 100%
publication_year = année de publication_date : 100%


- `open_model_weights` se déduit entièrement de `model_accessibility` : « Yes » pour les trois catégories « Open weights », « No » pour les autres.
- `training_compute_log10_flop` est le logarithme de `training_compute_flop`, et `publication_year` l'année de `publication_date`.

Ces colonnes sont redondantes, mais `open_model_weights` reste pratique pour la question 4.

### Catégories incohérentes

In [20]:
modeles['organization'].value_counts().head(12)

organization
OpenAI                 138
Alibaba                116
Google DeepMind        111
Google                  92
Meta AI                 77
Nvidia                  68
Microsoft               50
DeepMind                45
ByteDance               43
Mistral AI              41
Stanford University     36
DeepSeek                36
Name: count, dtype: int64

In [21]:
modeles.loc[modeles["organization"].str.contains("Google|DeepMind", na=False), "organization"].value_counts()

organization
Google DeepMind                                                                                                                                                                                                                                                                         111
Google                                                                                                                                                                                                                                                                                   92
DeepMind                                                                                                                                                                                                                                                                                 45
Google Brain                                                                                                                           

In [22]:
modeles['organization_type'].value_counts().head(8)

organization_type
Industry                        1956
Academia                         818
Academia;Industry                273
Industry;Academia                252
Academia;Research collective      29
Research collective               29
Industry;Government               26
Government                        24
Name: count, dtype: int64

In [23]:
modeles['country'].value_counts().head(6)

country
United States of America                             1232
China                                                 604
United States of America,United States of America     218
United Kingdom                                        131
China,China                                            71
Canada                                                 67
Name: count, dtype: int64

Ce sont les vrais défauts du jeu :

- **Organisations** : 1 318 libellés différents, avec la même entité écrite de plusieurs façons. Google apparaît sous « Google DeepMind », « Google », « DeepMind », « Google Brain », « Google Research » et des combinaisons comme « Google DeepMind,Google Research ».
- **Plusieurs valeurs dans une cellule** : séparées par des virgules (organisations, pays) ou des points-virgules (types d'organisation, domaines).
- **Ordre et répétitions** : « Academia;Industry » et « Industry;Academia » désignent la même chose, et des pays sont répétés dans la même cellule (« United States of America,United States of America » : 218 modèles).

### Doublons

In [24]:
colonnes_sans_nom = modeles.columns.drop("model_name")

print(f"Noms de modèles en double : {modeles['model_name'].duplicated().sum()}")
print(f"Lignes identiques hors nom : {modeles.duplicated(subset=colonnes_sans_nom).sum()}")

Noms de modèles en double : 0


Lignes identiques hors nom : 62


In [25]:
modeles_jumeaux = modeles[modeles.duplicated(subset=colonnes_sans_nom, keep=False)]
modeles_jumeaux.sort_values(["organization", "publication_date"])[["model_name", "organization", "publication_date", "parameters"]].head(6)

,model_name,organization,publication_date,parameters
3112,Firefly Image 4,Adobe,2025-04-24,NaN
3113,Firefly Image 4 Ultra,Adobe,2025-04-24,NaN
1996,Qwen Plus,Alibaba,2024-02-06,NaN
1997,Qwen-Turbo,Alibaba,2024-02-06,NaN
3261,Qwen3-235B-A22B (Jul 2025),Alibaba,2025-07-25,2.350000e+11
3262,Qwen3-235B-A22B-Thinking (Jul 2025),Alibaba,2025-07-25,2.350000e+11


Chaque nom de modèle est unique, mais 62 lignes sont identiques à une autre sur tout le reste : ce sont des variantes d'un même modèle (Qwen3 et sa version « Thinking », Firefly Image 4 et sa version « Ultra »…), publiées le même jour avec les mêmes caractéristiques. Ce sont des modèles distincts : on les garde, mais il faudra s'en souvenir en comptant les sorties par jour.

## 4. Constats et décisions de nettoyage

| Jeu | Constat | Décision prévue dans 02 |
|---|---|---|
| Hacker News | `story_type` constante, `created_at_i` et `year` redondantes | supprimer les trois colonnes |
| Hacker News | 3 466 articles sans `url` ni `domain` (messages « Ask HN ») | garder, signaler l'absence par une catégorie |
| Hacker News | 949 titres et 2 000 URL republiés | garder chaque publication, compter les sujets distincts quand c'est pertinent |
| Hacker News | 2 657 URL avec paramètres, dont des identifiants AWS expirés | retirer les paramètres de requête des URL |
| Hacker News | 2026 couverte jusqu'au 24 avril seulement | comparer les années en part, pas en volume |
| Epoch AI | 12 modèles sans date de publication | les retirer des analyses dans le temps |
| Epoch AI | `open_model_weights`, `training_compute_log10_flop`, `publication_year` redondantes | garder `open_model_weights`, supprimer les deux autres |
| Epoch AI | 1 318 libellés d'organisation, la même entité sous plusieurs noms | regrouper les grands labos sous un nom unique |
| Epoch AI | plusieurs valeurs par cellule, répétées ou dans un ordre différent | dédoublonner et trier les valeurs de chaque cellule |
| Epoch AI | 61 % de puissance de calcul et 94 % de coût manquants | ne pas imputer : analyser seulement les modèles renseignés et le dire |
| Epoch AI | 62 lignes identiques hors nom | garder : ce sont des variantes distinctes |